%md
#---------------------------------------------------------------------------------
#**SILVER**

In [0]:
%sql
CREATE TABLE IF NOT EXISTS electrocasa.silver.tracking_envios
AS
SELECT DISTINCT
    tracking_id,
    pedido_id,
    courier,

    CASE
        WHEN LOWER(TRIM(estado_entrega)) IN ('en_camino', 'en camino', 'en_transito')
            THEN 'en_transito'

        WHEN LOWER(TRIM(estado_entrega)) = 'pendiente'
            THEN 'pendiente'

        WHEN LOWER(TRIM(estado_entrega)) = 'entregado'
            THEN 'entregado'

        WHEN LOWER(TRIM(estado_entrega)) = 'devuelto'
            THEN 'devuelto'

        ELSE LOWER(TRIM(estado_entrega))
    END AS estado_entrega,

    sucursal_origen,
    fecha_actualizacion,
    fecha_ingestion,
    sistema_origen,
    batch_id

FROM electrocasa.bronze.tracking_envios
WHERE fecha_actualizacion IS NOT NULL;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS electrocasa.silver.tracking_envios_cuarentena (
    tracking_id STRING,
    pedido_id STRING,
    courier STRING,
    estado_entrega STRING,
    sucursal_origen STRING,
    fecha_actualizacion DATE,
    fecha_ingestion TIMESTAMP,
    sistema_origen STRING,
    batch_id STRING,
    motivo_rechazo STRING,
    fecha_rechazo TIMESTAMP
);

In [0]:
%sql
INSERT INTO electrocasa.silver.tracking_envios_cuarentena
SELECT
    tracking_id,
    pedido_id,
    courier,
    estado_entrega,
    sucursal_origen,
    fecha_actualizacion,
    fecha_ingestion,
    sistema_origen,
    batch_id,
    'fecha_actualizacion_nula' AS motivo_rechazo,
    current_timestamp() AS fecha_rechazo
FROM electrocasa.bronze.tracking_envios
WHERE fecha_actualizacion IS NULL;

In [0]:
%sql
SELECT *
FROM electrocasa.silver.tracking_envios_cuarentena
LIMIT 10;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS electrocasa.silver.catalogo_silver (
    producto_id STRING,
    nombre_producto STRING,
    categoria STRING,
    marca STRING,
    precio_lista DOUBLE,
    fecha_ingestion TIMESTAMP,
    sistema_origen STRING,
    batch_id STRING
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS electrocasa.silver.catalogo_quarantine (
    producto_id STRING,
    nombre_producto STRING,
    categoria STRING,
    marca STRING,
    precio_lista DOUBLE,
    fecha_ingestion TIMESTAMP,
    sistema_origen STRING,
    batch_id STRING,
    motivo_rechazo STRING,
    fecha_rechazo TIMESTAMP
);